# LoRA 微调 DeepSeek-R1-Distill-Llama-8B（Dolly 15k）
运行环境：Snowflake Notebook（Container Runtime）+ GPU 计算池 `DS_LORA_GPU_POOL`（GPU_NV_S，1×NVIDIA A10G 24GB）

流程：① 检查 GPU → ② 安装 PEFT → ③ 载入 Dolly 15k 并写入 Snowflake 表 → ④ 划分训练/评估集 → ⑤ 从 Model Registry 取出模型权重 → ⑥ 加载基础模型并做基线评估 → ⑦ 配置 LoRA → ⑧ 训练 → ⑨ 微调后评估与对比 → ⑩ 保存 LoRA 适配器到 Stage

In [ ]:
# ① 检查 GPU 与基础环境
import subprocess, sys, torch, platform
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("Python:", platform.python_version(), "| torch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
print(subprocess.run(["df", "-h", "/tmp", "/"], capture_output=True, text=True).stdout)

In [ ]:
# ② 通过 Snowflake 内置 PyPI 仓库（snowflake.snowpark.pypi_shared_repository）安装 PEFT
!pip install -q -U "peft>=0.18" accelerate
import sys; [sys.modules.pop(k) for k in list(sys.modules) if k.startswith('peft')]  # 清除失败的旧版导入缓存
import transformers, peft, accelerate
print("transformers", transformers.__version__, "| peft", peft.__version__, "| accelerate", accelerate.__version__)

In [ ]:
# ③ 载入 Dolly 15k（databricks/databricks-dolly-15k，已上传到当前 Workspace）并写入 Snowflake 表
import glob, os, pandas as pd
from snowflake.snowpark.context import get_active_session
session = get_active_session()
cands = glob.glob("databricks-dolly-15k.jsonl") or glob.glob("**/databricks-dolly-15k.jsonl", recursive=True) or glob.glob("/workspace/**/databricks-dolly-15k.jsonl", recursive=True)
print("当前目录:", os.getcwd())
DATA_PATH = cands[0]
df = pd.read_json(DATA_PATH, lines=True)
print("数据文件:", DATA_PATH, "| 总行数:", len(df))
print(df["category"].value_counts())
session.sql("USE SCHEMA LORA_DEMO.DOLLY").collect()
session.write_pandas(df.rename(columns=str.upper), "DOLLY_15K_RAW", auto_create_table=True, overwrite=True)
print(session.sql("SELECT CATEGORY, COUNT(*) AS N FROM LORA_DEMO.DOLLY.DOLLY_15K_RAW GROUP BY 1 ORDER BY 2 DESC").to_pandas())

In [ ]:
# ④ 构造提示词并划分训练 / 评估集（演示规模：800 条训练、100 条评估）
def build_user(row):
    ctx = (row["context"] or "").strip()
    return row["instruction"].strip() + (("\n\nContext:\n" + ctx) if ctx else "")
df["user"] = df.apply(build_user, axis=1)
df = df[(df["response"].str.len() > 0)].reset_index(drop=True)
shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
eval_df  = shuffled.iloc[:100].copy()
train_pool = shuffled.iloc[100:].copy()
print("评估集:", len(eval_df), "| 可用训练样本:", len(train_pool))
session.write_pandas(eval_df[["instruction","context","response","category"]].rename(columns=str.upper), "DS_LORA_EVAL", auto_create_table=True, overwrite=True)

In [ ]:
# ⑤ 从 Snowflake Model Registry 取出 DeepSeek-R1-Distill-Llama-8B 权重（试用账户无外网，权重由 Snowsight“Import model”导入）
import os, time
MODEL_URI = "snow://model/LORA_DEMO.DOLLY.DEEPSEEK_R1_DISTILL_LLAMA_8B/versions/V1/model/models/DEEPSEEK_R1_DISTILL_LLAMA_8B/model/"
LOCAL_DIR = "/tmp/ds_r1_llama8b"
os.makedirs(LOCAL_DIR, exist_ok=True)
t0 = time.time()
res = session.file.get(MODEL_URI, LOCAL_DIR, pattern=r".*\.(json|safetensors)")
print(f"下载完成：{len(res)} 个文件，用时 {time.time()-t0:.0f}s")
for root, _, files in os.walk(LOCAL_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); print(f"{os.path.getsize(p)/1e9:8.2f} GB  {p}")
MODEL_DIR = os.path.dirname(glob.glob(LOCAL_DIR + "/**/config.json", recursive=True)[0])
print("MODEL_DIR =", MODEL_DIR)

In [ ]:
# ⑥ 加载基础模型（bf16，直接放到 GPU）
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, device_map={"": 0}, low_cpu_mem_usage=True)
model.config.use_cache = False
n_params = sum(p.numel() for p in model.parameters())
print(f"基础模型参数量: {n_params/1e9:.2f} B | 显存占用: {torch.cuda.memory_allocated()/1e9:.1f} GB")
gen_prefix = tok.apply_chat_template([{"role":"user","content":"hi"}], tokenize=False, add_generation_prompt=True)
print("聊天模板（生成前缀）:", repr(gen_prefix))

In [ ]:
# ⑦ 构建训练样本：prompt 部分 label 置为 -100，只对回答计算损失
# 修正：transformers 5 对该 Llama-3 风格分词器会丢失 ByteLevel 解码器，这里直接用 tokenizer.json 重建
import json
from transformers import PreTrainedTokenizerFast
_cfg = json.load(open(f"{MODEL_DIR}/tokenizer_config.json"))
_tmpl = tok.chat_template
tok = PreTrainedTokenizerFast(tokenizer_file=f"{MODEL_DIR}/tokenizer.json", bos_token=tok.bos_token, eos_token=tok.eos_token, pad_token=tok.eos_token)
tok.chat_template = _cfg.get("chat_template") or _tmpl
print("分词往返校验:", repr(tok.decode(tok("Hello world, Snowflake LoRA!")["input_ids"], skip_special_tokens=True)))
gen_prefix = tok.apply_chat_template([{"role":"user","content":"hi"}], tokenize=False, add_generation_prompt=True)
print("聊天模板（生成前缀）:", repr(gen_prefix))
MAX_LEN = 512
THINK_OPEN = gen_prefix.endswith("<think>\n")
def encode(user, response):
    prompt = tok.apply_chat_template([{"role":"user","content":user}], tokenize=False, add_generation_prompt=True)
    answer = ("\n</think>\n\n" if THINK_OPEN else "") + response.strip() + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100]*len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "labels": labels, "full_len": len(p_ids)+len(a_ids)}
train_rows = [encode(u, r) for u, r in zip(train_pool["user"], train_pool["response"])]
train_rows = [x for x in train_rows if x["full_len"] <= MAX_LEN][:800]
eval_rows  = [encode(u, r) for u, r in zip(eval_df["user"], eval_df["response"])]
import numpy as np
print("训练样本:", len(train_rows), "| 平均 token 数:", int(np.mean([len(x['input_ids']) for x in train_rows])))
print("评估样本:", len(eval_rows))
eval_rows = [x for x in eval_rows if x["full_len"] <= MAX_LEN]  # 过滤超长样本，避免答案被截断导致 NaN
print("评估样本(过滤超长后):", len(eval_rows))

In [ ]:
# ⑧ 基线评估：评估集平均损失 / 困惑度 + 3 条示例生成
import math
@torch.no_grad()
def eval_loss(m, rows):
    m.eval(); tot, n = 0.0, 0
    for x in rows:
        ids = torch.tensor([x["input_ids"]], device="cuda"); lab = torch.tensor([x["labels"]], device="cuda")
        out = m(input_ids=ids, labels=lab)
        k = (lab[:,1:] != -100).sum().item(); tot += out.loss.item()*k; n += k
    return tot/n
@torch.no_grad()
def generate(m, user, max_new_tokens=200):
    m.eval()
    prompt = tok.apply_chat_template([{"role":"user","content":user}], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    m.config.use_cache = True
    out = m.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    m.config.use_cache = False
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
SAMPLES = eval_df.head(3)
base_loss = eval_loss(model, eval_rows)
print(f"[微调前] 评估集 loss = {base_loss:.4f} | 困惑度 = {math.exp(base_loss):.2f}")
base_gen = [generate(model, u) for u in SAMPLES["user"]]
for u, g in zip(SAMPLES["user"], base_gen):
    print("="*100); print("指令:", u[:300]); print("-"*40, "微调前输出"); print(g[:800])

In [ ]:
# ⑨ 配置 LoRA：冻结 8B 原始权重，只在注意力和 MLP 投影层注入低秩矩阵
from peft import LoraConfig, get_peft_model
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print(lora_cfg)

In [ ]:
# ⑩ 训练：1 epoch，batch=1，梯度累积 8（有效 batch 8），lr=2e-4，cosine 调度
import random, time
from transformers import get_cosine_schedule_with_warmup
EPOCHS, ACCUM, LR = 1, 8, 2e-4
random.seed(42); order = list(range(len(train_rows))); random.shuffle(order)
total_steps = math.ceil(len(order)*EPOCHS/ACCUM)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=5, num_training_steps=total_steps)
model.train(); log = []; run_loss = 0.0; step = 0; t0 = time.time()
for i, idx in enumerate(order):
    x = train_rows[idx]
    ids = torch.tensor([x["input_ids"]], device="cuda"); lab = torch.tensor([x["labels"]], device="cuda")
    loss = model(input_ids=ids, labels=lab).loss / ACCUM
    loss.backward(); run_loss += loss.item()
    if (i+1) % ACCUM == 0 or i+1 == len(order):
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
        log.append({"step": step, "loss": run_loss, "lr": sched.get_last_lr()[0], "elapsed_s": time.time()-t0})
        if step % 5 == 0 or step == 1:
            print(f"step {step:3d}/{total_steps} | loss {run_loss:.4f} | lr {sched.get_last_lr()[0]:.2e} | {time.time()-t0:6.0f}s | 显存峰值 {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
        run_loss = 0.0
print(f"训练完成：{step} 个优化步，用时 {(time.time()-t0)/60:.1f} 分钟")

In [ ]:
# ⑪ 训练损失曲线
import matplotlib.pyplot as plt
log_df = pd.DataFrame(log)
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(log_df["step"], log_df["loss"], label="train loss")
ax.plot(log_df["step"], log_df["loss"].rolling(10, min_periods=1).mean(), label="moving avg (10)")
ax.set_xlabel("optimizer step"); ax.set_ylabel("loss"); ax.set_title("LoRA fine-tuning DeepSeek-R1-Distill-Llama-8B on Dolly"); ax.legend(); ax.grid(alpha=.3)
plt.show()

In [ ]:
# ⑫ 微调后评估与对比
ft_loss = eval_loss(model, eval_rows)
print(f"[微调前] 评估集 loss = {base_loss:.4f} | 困惑度 = {math.exp(base_loss):.2f}")
print(f"[微调后] 评估集 loss = {ft_loss:.4f} | 困惑度 = {math.exp(ft_loss):.2f}")
ft_gen = [generate(model, u) for u in SAMPLES["user"]]
for u, ref, b, f in zip(SAMPLES["user"], SAMPLES["response"], base_gen, ft_gen):
    print("="*100); print("指令:", u[:300]); print("-"*30, "参考答案 (Dolly)"); print(ref[:500])
    print("-"*30, "微调前"); print(b[:600]); print("-"*30, "LoRA 微调后"); print(f[:600])

In [ ]:
# ⑬ 保存 LoRA 适配器（仅几十 MB）并上传到 Snowflake Stage，记录训练指标
ADAPTER_DIR = "/tmp/ds_r1_llama8b_dolly_lora"
model.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR)
for f in sorted(os.listdir(ADAPTER_DIR)): print(f"{os.path.getsize(os.path.join(ADAPTER_DIR,f))/1e6:8.2f} MB  {f}")
session.sql("CREATE STAGE IF NOT EXISTS LORA_DEMO.DOLLY.DS_LORA_ADAPTER_STAGE").collect()
for f in os.listdir(ADAPTER_DIR):
    session.file.put(os.path.join(ADAPTER_DIR, f), "@LORA_DEMO.DOLLY.DS_LORA_ADAPTER_STAGE/dolly_lora_r16/", auto_compress=False, overwrite=True)
log_df["BASE_EVAL_LOSS"] = base_loss; log_df["FT_EVAL_LOSS"] = ft_loss
session.write_pandas(log_df.rename(columns=str.upper), "DS_LORA_TRAIN_LOG", auto_create_table=True, overwrite=True)
print(session.sql("LIST @LORA_DEMO.DOLLY.DS_LORA_ADAPTER_STAGE").to_pandas().iloc[:, :2].to_string())

## 收尾
- LoRA 适配器保存在 `@LORA_DEMO.DOLLY.DS_LORA_ADAPTER_STAGE/dolly_lora_r16/`，训练日志在 `LORA_DEMO.DOLLY.DS_LORA_TRAIN_LOG`。
- 演示结束后请关闭 Notebook 服务，并执行 `ALTER COMPUTE POOL DS_LORA_GPU_POOL SUSPEND;` 避免继续消耗额度。